In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:53:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df = spark.range(0,1000000).withColumn("value",col("id")%1000)

In [3]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [18]:
opti_df=df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [8]:
opti_df.explain()

== Physical Plan ==
*(1) Project [id#5L, (id#5L % 1000) AS value#7L]
+- *(1) Filter (((id#5L % 1000) > 500) AND (id#5L < 5000000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [9]:
stTime=time.time()
count_uncached=opti_df.count()
enTime=time.time()
print(f"1.Unoptimized Execution | Count:{count_uncached} | Time :{round(enTime-stTime,4)} seconds")

1.Optimized Execution | Count:499000 | Time :0.7197 seconds


In [16]:
opti_df.cache()
stTime=time.time()
count_cached=opti_df.count()
enTime=time.time()
print(f"2.First Cached Execution | Count:{count_cached} | Time :{round(enTime-stTime,4)} seconds")

2.First Cached Execution | Count:499000 | Time :0.1373 seconds


26/06/13 06:11:13 WARN CacheManager: Asked to cache already cached data.


In [17]:
stTime=time.time()
count_cached2=opti_df.count()
enTime=time.time()
print(f"3.Second Cached Execution | Count:{count_cached2} | Time :{round(enTime-stTime,4)} seconds")

3.Second Cached Execution | Count:499000 | Time :0.1403 seconds


In [27]:
from pyspark import StorageLevel
df.persist(StorageLevel.DISK_ONLY)
stTime=time.time()
count_cached3=opti_df.count()
enTime=time.time()
print(f"4.Unoptimized Execution | Count:{count_cached3} | Time :{round(enTime-stTime,4)} seconds")

4.Unoptimized Execution | Count:499000 | Time :0.1701 seconds


26/06/13 06:26:13 WARN CacheManager: Asked to cache already cached data.


In [ ]:
spark.